# Exercice : Intégration de l'API eBay

**Client Credentials Grant — Browse API**

Documentation : [OAuth Client Credentials Grant](https://developer.ebay.com/api-docs/static/oauth-client-credentials-grant.html) | [Outil de test eBay](https://developer.ebay.com/my/api_test_tool?index=0&env=production)

## Installation des dépendances

Exécutez cette cellule une seule fois.

In [ ]:
%pip install requests python-dotenv --quiet

## Configuration du logging

Toute la classe devra utiliser `logging`.

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

## Vérification des credentials

Créez un fichier `.env` à la racine du projet :

```
app_id=VotreAppID
dev_id=VotreDevID
client_secret=VotreClientSecret
```

> Ne commitez jamais votre fichier `.env` — ajoutez-le à votre `.gitignore`.

In [ ]:
import os
import dotenv

dotenv.load_dotenv()

for var in ["app_id", "dev_id", "client_secret"]:
    status = "chargé" if os.getenv(var) else "MANQUANT"
    logger.info("%s : %s", var, status)

---
## Partie 1 — Authentification OAuth

### 1.1 Obtenir un token d'accès

L'API eBay utilise le flux **Client Credentials Grant**. La requête doit être envoyée à :

```
POST https://api.ebay.com/identity/v1/oauth2/token
```

| Element | Valeur |
|---------|--------|
| Header `Authorization` | `Basic <base64(app_id:client_secret)>` |
| Header `Content-Type` | `application/x-www-form-urlencoded` |
| Body `grant_type` | `client_credentials` |
| Body `scope` | `https://api.ebay.com/oauth/api_scope` |

### 1.2 Gestion de l'expiration

La réponse contient `expires_in` (en secondes). Le token ne doit être renouvelé que lorsqu'il est expiré, et ce renouvellement doit être transparent pour le code appelant.

Réfléchissez à comment Python vous permet d'exposer cela proprement : comment rendre l'accès au token aussi simple que l'accès à un attribut, tout en cachant la logique de refresh ?

In [ ]:
import base64
import os
from datetime import datetime, timedelta

import requests
import dotenv

dotenv.load_dotenv()


class EbayApi:
    def __init__(self):
        # Charger les credentials depuis les variables d'environnement.
        # Lever une ValueError si l'un d'eux est manquant.
        # Initialiser les attributs nécessaires pour la gestion du token.
        pass

    # Réfléchissez aux méthodes à implémenter :
    #
    # - Comment obtenir un token depuis l'API eBay ?
    # - Comment exposer le token de façon transparente, avec refresh automatique ?
    # - Comment factoriser la logique commune à toutes vos requêtes HTTP GET ?
    # - Quelles méthodes constituent l'interface publique de la classe ?
    #
    # Utilisez logger.info / logger.error pour tracer les opérations importantes.
    # Aucun print n'est autorisé.

### Test Partie 1 — Token OAuth

Vérifiez que vous obtenez bien un token, et que le refresh automatique fonctionne.

In [ ]:
# Test : obtention du token
try:
    api = EbayApi()
    token = api.access_token
    logger.info("Token obtenu : %s...", token[:30])
except Exception as e:
    logger.error("Erreur : %s", e)

In [ ]:
# Test : refresh automatique
# Simulez une expiration du token et vérifiez qu'un nouveau est bien obtenu.
try:
    api = EbayApi()
    _ = api.access_token

    api._expiration_time = datetime.now() - timedelta(seconds=1)
    logger.info("Token marqué comme expiré")

    new_token = api.access_token
    logger.info("Nouveau token après refresh : %s...", new_token[:30])
except Exception as e:
    logger.error("Erreur : %s", e)

---
## Partie 2 — Requêtes à la Browse API

### 2.1 Recherche par mot-clé

Endpoint : `GET https://api.ebay.com/buy/browse/v1/item_summary/search?q=<query>`

Headers obligatoires :
```python
{
    "Authorization": f"Bearer {access_token}",
    "X-EBAY-C-MARKETPLACE-ID": "EBAY_FR",
}
```

Toutes vos requêtes GET partagent la même logique. Pensez à factoriser.

### 2.2 Récupération d'un article par URL

Implémentez une méthode pour effectuer un GET sur une URL complète. Exemple :
```
https://api.ebay.com/buy/browse/v1/item/v1%7C358311088969%7C626586956773
```

In [ ]:
# Test : recherche par mot-clé
import json

try:
    api = EbayApi()
    results = api.search_item("vélo électrique")
    logger.info("Nombre de résultats : %s", results.get("total", "N/A"))
    items = results.get("itemSummaries", [])
    if items:
        logger.info("Premier article : %s", items[0].get("title", "N/A"))
except Exception as e:
    logger.error("Erreur : %s", e)

In [ ]:
# Test : récupération par URL
try:
    api = EbayApi()
    item = api.search_item_ref(
        "https://api.ebay.com/buy/browse/v1/item/v1%7C358311088969%7C626586956773"
    )
    logger.info("Titre : %s", item.get("title", "N/A"))
    logger.info("Condition : %s", item.get("condition", "N/A"))
except Exception as e:
    logger.error("Erreur : %s", e)

---
## Partie 3 — Gestion des erreurs

Vérifiez que votre classe lève bien les bonnes exceptions dans les cas limites.

In [ ]:
# Test : credentials manquants
original = os.environ.pop("app_id", None)

try:
    api_broken = EbayApi()
    logger.warning("Aucune exception levée — à corriger")
except ValueError as e:
    logger.info("ValueError levée correctement : %s", e)
except Exception as e:
    logger.warning("Exception levée mais pas une ValueError : %s - %s", type(e).__name__, e)
finally:
    if original:
        os.environ["app_id"] = original

---
## Sauvegarde des résultats

Sauvegardez les résultats en JSON.

In [ ]:
import os
import json

os.makedirs("data/json_response", exist_ok=True)

try:
    api = EbayApi()

    search_results = api.search_item("vélo électrique")
    with open("data/json_response/search_response.json", "w", encoding="utf-8") as f:
        json.dump(search_results, f, indent=4, ensure_ascii=False)
    logger.info("Résultats sauvegardés dans data/json_response/search_response.json")

    item = api.search_item_ref(
        "https://api.ebay.com/buy/browse/v1/item/v1%7C358311088969%7C626586956773"
    )
    with open("data/json_response/search_item_ref_response.json", "w", encoding="utf-8") as f:
        json.dump(item, f, indent=4, ensure_ascii=False)
    logger.info("Article sauvegardé dans data/json_response/search_item_ref_response.json")

except Exception as e:
    logger.error("Erreur lors de la sauvegarde : %s", e)

---

> Conseil : utilisez l'[outil de test eBay](https://developer.ebay.com/my/api_test_tool?index=0&env=production) pour vérifier vos credentials et visualiser les réponses avant de commencer à coder.